In [1]:
import os 
import pandas as pd
import numpy as np

os.chdir("../")
os.getcwd()

'c:\\Users\\leoco\\Documents\\cours\\M2_MOSEF\\time_series_deep_learning\\Store_Sales_Time_Series_Forecasting'

In [2]:
from src.create_dataset import DatasetProcessor

In [3]:
main_df = pd.read_csv('data/main_df.csv', parse_dates=['date'])
stores_df = pd.read_csv('data/stores.csv')
oil_df = pd.read_csv('data/oil.csv', parse_dates=['date'])
holidays_df = pd.read_csv('data/holidays_events.csv', parse_dates=['date'])

In [4]:
main_df.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [5]:
stores_df.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [6]:
oil_df.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [7]:
holidays_df

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


In [8]:
# 8 dernières semaines pour le test (56 jours)
last_date = main_df['date'].max()
test_start_date = last_date - pd.Timedelta(days=55)
test_raw = main_df[main_df['date'] >= test_start_date]

train_raw = main_df[main_df['date'] < test_start_date]

In [9]:
processor = DatasetProcessor(stores_df, oil_df, holidays_df)

# Apprentissage sur le Train (calcul des quartiles et moyennes)
processor.fit(train_raw)

# Transformation du Train et du Test
train = processor.transform(train_raw, is_test=False)
test = processor.transform(test_raw, is_test=True)

Calcul des statistiques de référence sur le Train...
Stats enregistrées pour 33 familles.

Mode TRAIN activé
Nombre de lignes avant traitement : 2901096
Préparation des données du pétrole du 2013-01-01 au 2017-06-20...
Traitement des jours fériés et priorités géographiques...
Calcul des variables temporelles et cycliques...
Génération des lags et fenêtres glissantes...
Analyse de la mobilité des stocks et inactivité...
Nombre de lignes après traitement : 2901096
Optimisation de la memoire en cours...
Reduction memoire: 1029.2MB -> 395.6MB (61.6%)

Mode TEST détecté
Nombre de lignes avant traitement : 99792
Ajout de l'historique du train pour les lags...
Préparation des données du pétrole du 2016-06-21 au 2017-08-15...
Traitement des jours fériés et priorités géographiques...
Calcul des variables temporelles et cycliques...
Génération des lags et fenêtres glissantes...
Analyse de la mobilité des stocks et inactivité...
Suppression de l'historique de préchauffage terminée.
Nombre de lign

In [10]:
train.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,...,sales_lag_56,sales_lag_364,rolling_mean_7,rolling_mean_28,log_sales,rolling_std_7,is_store_closed,is_family_unactive,is_store_closed_lag_1,is_family_unactive_lag_1
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,0.000000,NaN,1,0,0.0,0.0
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,1.098612,NaN,0,1,1.0,0.0
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,1.386294,NaN,0,0,0.0,1.0
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,1.386294,NaN,0,0,0.0,0.0
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,...,NaN,NaN,NaN,NaN,1.791759,NaN,0,0,0.0,0.0


In [11]:
train.isnull().sum()

id                                0
date                              0
store_nbr                         0
family                            0
sales                             0
onpromotion                       0
city                              0
state                             0
store_type                        0
cluster                           0
oil_price                         0
oil_price_trend_30                0
oil_price_trend_60                0
oil_price_lag_8w                  0
oil_price_lag_8w_trend_30         0
oil_price_lag_8w_trend_60         0
holiday_type                      0
is_holiday_local                  0
is_holiday_regional               0
is_holiday_national               0
day_of_week                       0
month                             0
year                              0
is_weekend                        0
is_payday                         0
month_sin                         0
month_cos                         0
day_sin                     

In [12]:
train.dtypes

id                                    int32
date                         datetime64[ns]
store_nbr                              int8
family                             category
sales                               float32
onpromotion                           int16
city                               category
state                              category
store_type                         category
cluster                                int8
oil_price                           float32
oil_price_trend_30                  float32
oil_price_trend_60                  float32
oil_price_lag_8w                    float32
oil_price_lag_8w_trend_30           float32
oil_price_lag_8w_trend_60           float32
holiday_type                       category
is_holiday_local                       int8
is_holiday_regional                    int8
is_holiday_national                    int8
day_of_week                            int8
month                                  int8
year                            

In [13]:
train.select_dtypes(include="category")

,family,city,state,store_type,holiday_type
0,AUTOMOTIVE,Quito,Pichincha,D,Holiday
1782,AUTOMOTIVE,Quito,Pichincha,D,Normal Day
3564,AUTOMOTIVE,Quito,Pichincha,D,Normal Day
5346,AUTOMOTIVE,Quito,Pichincha,D,Normal Day
7128,AUTOMOTIVE,Quito,Pichincha,D,Normal Day
...,...,...,...,...,...
2941949,SEAFOOD,El Carmen,Manabi,C,Normal Day
2943731,SEAFOOD,El Carmen,Manabi,C,Normal Day
2945513,SEAFOOD,El Carmen,Manabi,C,Normal Day
2947295,SEAFOOD,El Carmen,Manabi,C,Normal Day


In [14]:
train.to_parquet('data/train_sales_data.parquet')
test.to_parquet('data/test_sales_data.parquet')